# 04 CNN 图像分类

## 本节目标

- 理解图像张量的 `[N, C, H, W]` 格式。
- 使用卷积层和池化层学习局部特征。
- 在小型图像数据集上训练 CNN。
- 通过 loss 曲线和预测结果观察模型表现。

## 背景问题

本节关注的问题是：图像中的局部结构如何被模型利用？

CNN 用卷积核在图像上滑动，学习局部模式。这个概念可以从代码角度理解为：卷积层接收四维图像张量，并输出新的特征图。

## 核心概念

- Channel：图像通道，灰度图为 1，RGB 图为 3。
- Convolution：通过共享参数学习局部特征。
- Pooling：降低空间尺寸，减少后续计算量。
- Flatten：把特征图转换成分类层可接收的一维特征。

## 最小代码示例

下面先加载小型手写数字图像，并检查输入 shape。初学者容易在这里混淆 `[N, H, W]` 和 `[N, C, H, W]`。

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

digits = load_digits()
images = digits.images.astype(np.float32) / 16.0
labels = digits.target.astype(np.int64)

x_train, x_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)

x_train = torch.tensor(x_train).unsqueeze(1)
x_test = torch.tensor(x_test).unsqueeze(1)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

print("train image shape:", x_train.shape)

plt.figure(figsize=(6, 2))
for i in range(6):
    plt.subplot(1, 6, i + 1)
    plt.imshow(x_train[i, 0], cmap="gray")
    plt.title(str(y_train[i].item()))
    plt.axis("off")
plt.tight_layout()
plt.show()

## 完整实验

这里的关键不是搭建很深的 CNN，而是观察卷积、池化和全连接分类头如何组成一个完整图像分类模型。

### 训练与可视化

下面训练一个小型 CNN，并画出 loss 曲线和若干预测结果。调试时可以优先打印卷积输出的 shape，再确认分类层的 `in_features`。

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)
model = TinyCNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []

for epoch in range(20):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
    losses.append(total_loss / len(train_loader.dataset))

model.eval()
with torch.no_grad():
    test_logits = model(x_test)
    test_pred = test_logits.argmax(dim=1)
    test_acc = (test_pred == y_test).float().mean().item()

print(f"final loss={losses[-1]:.4f}, test accuracy={test_acc:.3f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("CNN training loss")
plt.xlabel("Epoch")
plt.ylabel("Cross entropy")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 2.5))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    plt.imshow(x_test[i, 0], cmap="gray")
    plt.title(f"p:{test_pred[i].item()}\ny:{y_test[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 实验观察

从运行结果可以看到，loss 曲线明显下降，测试准确率通常能达到较高水平。由于图像较小且结构清晰，小型 CNN 也能快速学到有效模式。少量错误通常出现在形状相近的数字上。

## 常见错误

- 忘记添加 channel 维度，输入仍是 `[N, H, W]`。
- `nn.Linear` 的输入特征数计算错误。
- 标签不是 `torch.long`。
- 测试阶段忘记使用 `model.eval()` 和 `torch.no_grad()`。

初学者容易在这里混淆图像尺寸和特征维度，建议每经过一个卷积或池化模块就打印一次 shape。

In [ ]:
sample = x_train[:4]
with torch.no_grad():
    feature = model.features(sample)
print("input shape:", sample.shape)
print("feature shape before classifier:", feature.shape)

## 概念问答

**Q：为什么 CNN 不直接把图像 flatten？**  
A：直接 flatten 会弱化空间邻接结构，卷积层能更自然地利用局部模式。

**Q：池化一定必须使用吗？**  
A：不是必须，但它常用于降低尺寸、减少计算，并提升一定的位置鲁棒性。

**Q：为什么输入要有 channel 维？**  
A：卷积层需要知道每个样本有多少输入通道，即使灰度图也要显式保留 `C=1`。

## 小结

CNN 的关键是用卷积和池化逐步学习空间特征。写 CNN 时最需要盯紧输入维度、特征图尺寸和分类层输入大小。